# Fundamentals 02 - Skill API

Objetivo: construir una `Skill` como paquete reutilizable de tools, prompts, contratos, policy y metadata. Una skill no ejecuta modelos por si sola; empaqueta capacidades para que un agente, sistema o grafo las consuma de forma explicita.

In [ ]:
import agentic_systems as toolkit

PRETTY = False  # True usa Rich; False deja salida estable para notebooks y CI.

## Escenario didactico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```


## Parámetros de `RunPolicy`

`RunPolicy` declara como debe comportarse una ejecucion antes de llamar al agente o al runtime. No es metadata decorativa: limita loops, define reparacion, controla trazas y hace que el resultado sea evaluable.

| Parametro | Que controla | Uso recomendado |
|---|---|---|
| `max_turns` | Numero maximo de turnos internos del agente. | Mantenerlo bajo en notebooks para evitar loops largos. |
| `max_tool_calls` | Numero maximo de llamadas a tools. | Declararlo cuando el ejercicio espera tools concretas. |
| `max_tokens` | Limite de tokens del modelo cuando el provider lo soporta. | util en providers LM; puede quedar `None` en `python-runtime`. |
| `temperature` | Aleatoriedad del modelo. | `0.0` para tutoriales reproducibles; `None` delega al provider. |
| `tool_choice` | Estrategia de seleccion de tools, por ejemplo `auto`. | `auto` cuando el agente decide; explicito cuando quieres forzar una tool. |
| `repair` | Permite reparacion automatica de salidas o tool calls invalidas. | `True` para UX robusta; `False` si quieres ver fallos crudos. |
| `max_repairs` | Maximo de intentos de reparacion. | `1` o `2` en tutoriales para mostrar control sin ocultar errores. |
| `finalize` | Que hacer al agotar turnos, por ejemplo `on_max_turns`. | Mantenerlo explicito en agentes LM evaluables. |
| `trace` | Nivel de trazabilidad (`compact`, `debug`, etc.). | `compact` para notebooks; `debug` solo para diagnostico. |
| `strict` | Si el contrato debe aplicarse de forma estricta. | `True` para ensenar API y evitar ambiguedad. |


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

toolkit.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario didactico - visible")

## 1) Definir tools que formaran la skill

Una `Skill` agrupa tools ya existentes. La skill no reemplaza a la tool; le da contexto, contrato y una frontera reutilizable.

In [ ]:
@toolkit.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos numeros."""
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@toolkit.tool
def restar(a: int, b: int) -> dict:
    """Resta b a a."""
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@toolkit.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos numeros."""
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} * {b} = {a * b}"}


@toolkit.tool
def dividir(a: float, b: float) -> dict:
    """Divide a entre b con validacion explicita."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    return {"operation": "dividir", "result": a / b, "explanation": f"{a} / {b} = {a / b}"}

## 2) Crear la skill

La skill documenta que contiene y que politica espera. Esta metadata es serializable y no requiere credenciales ni provider cloud.

In [ ]:
calculator_contract = toolkit.AgentContract(
    must_call=["sumar", "restar", "multiplicar", "dividir"],
    completion="when_required_tools_satisfied",
)

calculator_policy = toolkit.RunPolicy(
    max_tool_calls=4,
    max_turns=4,
)

math_skill = toolkit.skill(
    name="fundamentals_math",
    version="1.0.0",
    description="Skill aritmetica reusable para el escenario didactico de fundamentals.",
    tools=[sumar, restar, multiplicar, dividir],
    prompts={
        "instructions": "Usa las tools aritmeticas y conserva procedimiento auditable.",
        "user_prompt": USER_PROMPT,
    },
    contracts={"default": calculator_contract.model_dump(mode="json")},
    policy=calculator_policy.model_dump(mode="json"),
    metadata={"domain": "tutorials", "runtime_safe": True},
)

toolkit.show(math_skill.info(), title="Skill info - serializable")

## 3) Validar la skill antes de usarla

`skill.check()` valida la definicion local: nombres duplicados, tools invalidas y errores de contrato de cada tool.

In [ ]:
skill_validation = math_skill.check()

toolkit.show({
    "ok": skill_validation.ok,
    "tool_names": math_skill.tool_names,
    "description": math_skill.describe(),
    "instructions": math_skill.instructions,
}, title="Skill check")

## 4) Usar prompts y tools desde la skill

La skill expone `prompt(...)`, `tool(...)` y `available_tools()` para que el consumidor no tenga que conocer variables internas del notebook.

In [ ]:
prompt_from_skill = math_skill.prompt("user_prompt")
selected_tool = math_skill.tool("sumar")
direct_result = selected_tool.run({"a": 10, "b": 20})

toolkit.show({
    "prompt_from_skill": prompt_from_skill,
    "available_tools": [tool.name for tool in math_skill.available_tools()],
    "direct_tool_result": direct_result.normalized(),
}, title="Skill como paquete reusable")

## 5) Consumir la skill desde un agente

El agente recibe `skills=[math_skill]`. Agentic Systems expande sus tools y conserva la skill como unidad conceptual.

In [ ]:
single_call_contract = toolkit.AgentContract(
    must_call=["sumar"],
    completion="when_required_tools_satisfied",
)

agent = toolkit.agent(
    name="skill_calculator_agent",
    instructions=math_skill.instructions,
    skills=[math_skill],
    contract=single_call_contract,
    policy=calculator_policy,
    engine="python-runtime",
)

result = agent.run({"tool": "sumar", "input": {"a": 10, "b": 20}}, mode="eval")
result.validation = result.validate(single_call_contract).to_dict()

toolkit.human_result(
    result,
    title="Human result - Agent usando Skill",
    expected_tools=toolkit.expect.exactly("sumar"),
    pretty=PRETTY,
)

## 6) Resolver el escenario didactico con la skill

Aqui no aparece una abstraccion nueva ni se llama a Bedrock: se ejecutan las tools empaquetadas por la skill con Python local y se proyecta el resultado con `human_result`.

In [ ]:
operation_trace = []
tool_results = []
value = 10

for tool_name, kwargs in [
    ("sumar", {"a": value, "b": 20}),
    ("restar", {"a": 30, "b": 9}),
    ("multiplicar", {"a": 21, "b": 4}),
    ("dividir", {"a": 84, "b": 2}),
]:
    output = math_skill.tool(tool_name).run(kwargs)
    value = output.data["result"]
    operation_trace.append(output.data)
    tool_results.append(output)

final = toolkit.final_answer(
    {
        "procedimiento": [item["explanation"] for item in operation_trace],
        "resultado_final": value,
    },
    schema=toolkit.output_schema(fields=REQUESTED_OUTPUTS),
    text="El resultado final es 42.",
)

skill_result = toolkit.compose_result(
    text="El resultado final es 42.",
    data=final,
    results=tool_results,
    mode="skill-pipeline",
    input=USER_PROMPT,
    meta={"skill": math_skill.info(), "operation_trace": operation_trace},
)

toolkit.human_result(
    skill_result,
    title="Human result - Escenario didactico con Skill",
    expected_tools=None,
    pretty=PRETTY,
)

## Lo importante

- `Skill` empaqueta tools, prompts, contracts, policy y metadata.
- `Skill` no depende de OpenAI, Bedrock, LangGraph ni Strands.
- `skill.check()` valida el paquete antes de conectarlo a agentes o sistemas.
- `agent(..., skills=[skill])` consume la skill sin perder las tools directas.
- El resultado humano sigue saliendo por `toolkit.human_result(...)`.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {
        "api": "toolkit.skill",
        "description": "Empaqueta tools, prompts, contratos, policy y metadata como una capacidad reusable.",
    },
    {
        "api": "Skill.info",
        "description": "Expone la skill como payload serializable para auditor?a y documentacion.",
    },
    {
        "api": "Skill.check",
        "description": "Valida la definicion local antes de conectarla a agentes o sistemas.",
    },
    {
        "api": "Skill.prompt / Skill.tool / Skill.available_tools",
        "description": "Permite consumir prompts y tools sin conocer variables internas del notebook.",
    },
    {
        "api": "agent(..., skills=[skill])",
        "description": "Demuestra como un agente consume una skill manteniendo contrato y policy explicitos.",
    },
]

toolkit.show({"notebook": "02_skill_api.ipynb", "api_coverage": api_coverage})

## Simbolos API explicados

Este notebook se alinea con `docs/API.md` y ensena estos simbolos publicos:

- `Skill`: Paquete de tools, instrucciones y contratos.
- `SkillManifest / LoadedSkill / load_skill`: API publica para skills cargables desde disco.
- `toolkit.compose_result`: Compone resultados de pipeline sin fabricar metadata falsa.
- `output_schema / final_answer`: Contrato de salida usado por una skill.
- `AgentContract / RunPolicy`: Contrato aplicado al agente que consume skill.

